In [3]:
import requests
import zipfile
import os

# --- STEP 1: CONFIGURATION ( for 2025) ---
url = "https://divvy-tripdata.s3.amazonaws.com/202501-divvy-tripdata.zip"
raw_dir = "../data/raw"
zip_name = "202501-divvy-tripdata.zip"
zip_path = os.path.join(raw_dir, zip_name)

# --- STEP 2: THE DOWNLOAD ---
print(f"1. Downloading {zip_name} (This might take 15-30 seconds)...")
response = requests.get(url)

if response.status_code == 200:
    # Save the zip file to your E: drive
    with open(zip_path, 'wb') as f:
        f.write(response.content)
    print("✅ Download Complete!")
    
    # --- STEP 3: THE UNZIP ---
    print("2. Unzipping the suitcase...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(raw_dir)
    print("✅ Unzip Complete! The CSV is ready.")
    
    # --- STEP 4: CLEANUP & VERIFICATION ---
    # We delete the zip file to save space on your hard drive!
    os.remove(zip_path)
    print("🧹 Cleaned up the leftover zip file.")
    
    print(f"\nFiles now sitting in your data folder: {os.listdir(raw_dir)}")
    
else:
    print(f"❌ Failed! The internet said no. Code: {response.status_code}")

1. Downloading 202501-divvy-tripdata.zip (This might take 15-30 seconds)...
✅ Download Complete!
2. Unzipping the suitcase...


PermissionError: [Errno 13] Permission denied: '..\\data\\raw\\202501-divvy-tripdata.csv'

In [ ]:
import pandas as pd

csv_path = "../data/raw/202501-divvy-tripdata.csv"
print("Loading data into Pandas... (This might take a few seconds)")

df = pd.read_csv(csv_path)
print("✅ Data Loaded! Here is what the data actually looks like:")

df.head()

Loading data into Pandas... (This might take a few seconds)
✅ Data Loaded! Here is what the data actually looks like:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member
1,013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member
2,EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member
3,EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member
4,7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138689 entries, 0 to 138688
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             138689 non-null  str    
 1   rideable_type       138689 non-null  str    
 2   started_at          138689 non-null  str    
 3   ended_at            138689 non-null  str    
 4   start_station_name  115837 non-null  str    
 5   start_station_id    115837 non-null  str    
 6   end_station_name    114616 non-null  str    
 7   end_station_id      114616 non-null  str    
 8   start_lat           138689 non-null  float64
 9   start_lng           138689 non-null  float64
 10  end_lat             138628 non-null  float64
 11  end_lng             138628 non-null  float64
 12  member_casual       138689 non-null  str    
dtypes: float64(4), str(9)
memory usage: 13.8 MB


In [6]:
#Convert the text columns into official "Datetime" objects
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

df['ride_length'] = df['ended_at'] - df['started_at']

print("✅ Dates converted! Here is your new calculated column:")
df[['started_at', 'ended_at', 'ride_length']].head()

✅ Dates converted! Here is your new calculated column:


,started_at,ended_at,ride_length
0,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,0 days 00:13:57.477000
1,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,0 days 00:05:04.344000
2,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,0 days 00:11:35.500000
3,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,0 days 00:03:34.233000
4,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,0 days 00:02:34.429000


In [7]:
# Extract the Day of the Week 
df['day_of_week'] = df['started_at'].dt.day_name()

# Convert ride_length to total minutes
df['ride_length_mins'] = df['ride_length'].dt.total_seconds() / 60

print("✅ Feature Engineering Complete! Here is the clean data:")
df[['started_at', 'day_of_week', 'ride_length_mins', 'member_casual']].head()

✅ Feature Engineering Complete! Here is the clean data:


,started_at,day_of_week,ride_length_mins,member_casual
0,2025-01-21 17:23:54.538,Tuesday,13.957950,member
1,2025-01-11 15:44:06.795,Saturday,5.072400,member
2,2025-01-02 15:16:27.730,Thursday,11.591667,member
3,2025-01-23 08:49:05.814,Thursday,3.570550,member
4,2025-01-16 08:38:32.338,Thursday,2.573817,member


In [8]:
# Find the "Ghost Rides" (Negative or exactly 0 minutes)
short_rides_error = df[df['ride_length_mins'] <= 0]

# Find the "Stolen/Lost Bikes" (Rides longer than 24 hours / 1440 minutes)
long_rides_error = df[df['ride_length_mins'] > 1440]

print(f"Bikes with negative/zero time (Errors): {len(short_rides_error)} rows")
print(f"Bikes missing for over 24 hours (Lost): {len(long_rides_error)} rows")

# peek at one of the broken rows to see what it looks like
if len(short_rides_error) > 0:
    print("\nHere is an example of a broken 'Negative/Zero Time' ride:")
    display(short_rides_error[['started_at', 'ended_at', 'ride_length_mins']].head(1))

Bikes with negative/zero time (Errors): 0 rows
Bikes missing for over 24 hours (Lost): 69 rows


In [9]:
# Record the starting number so we can double-check our math
original_count = len(df)

# The Filter: Keep only rides longer than 0 mins AND shorter than 1440 mins
df_clean = df[(df['ride_length_mins'] > 0) & (df['ride_length_mins'] <= 1440)]

# Calculate how many we dropped
cleaned_count = len(df_clean)
dropped_count = original_count - cleaned_count

print("🧹 Data Cleaning Complete!")
print(f"Original Data: {original_count} rows")
print(f"Removed:       {dropped_count} broken rows")
print(f"Clean Data:    {cleaned_count} perfect rows")

🧹 Data Cleaning Complete!
Original Data: 138689 rows
Removed:       69 broken rows
Clean Data:    138620 perfect rows


In [10]:
# Group the data by rider type, and calculate the AVERAGE ride length for each
summary_table = df_clean.groupby('member_casual')['ride_length_mins'].mean()

print("📊 Average Ride Length (in minutes):")
print(summary_table)

📊 Average Ride Length (in minutes):
member_casual
casual    11.753248
member     9.800749
Name: ride_length_mins, dtype: float64


In [11]:
import requests
import zipfile
import os
import time

# 1. We already have January ("01"), so let's list the remaining 11 months
months = ["02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
raw_dir = "../data/raw"

print("🚀 Starting the Master Ingestion Pipeline for 2025...\n")

# 2. Start the Loop! 
for month in months:
    # Build the exact URL for each month
    zip_name = f"2025{month}-divvy-tripdata.zip"
    url = f"https://divvy-tripdata.s3.amazonaws.com/{zip_name}"
    zip_path = os.path.join(raw_dir, zip_name)

    print(f"Fetching {month}/2025...")
    response = requests.get(url)

    # 3. If the download works, extract it and clean up
    if response.status_code == 200:
        with open(zip_path, 'wb') as f:
            f.write(response.content)
            
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(raw_dir)
            
        os.remove(zip_path) # Delete the heavy zip file
        print(f"✅ Month {month} Complete!")
    else:
        print(f"❌ Failed to find data for month {month}. Error: {response.status_code}")

    # 4. Pause for 2 seconds to be polite to the Divvy servers so they don't block us
    time.sleep(2)

print("\n🎉 PIPELINE FINISHED! All 12 months of 2025 data are safely on your hard drive.")

🚀 Starting the Master Ingestion Pipeline for 2025...

Fetching 02/2025...
✅ Month 02 Complete!
Fetching 03/2025...
✅ Month 03 Complete!
Fetching 04/2025...
✅ Month 04 Complete!
Fetching 05/2025...
✅ Month 05 Complete!
Fetching 06/2025...
✅ Month 06 Complete!
Fetching 07/2025...
✅ Month 07 Complete!
Fetching 08/2025...
✅ Month 08 Complete!
Fetching 09/2025...
✅ Month 09 Complete!
Fetching 10/2025...
✅ Month 10 Complete!
Fetching 11/2025...
✅ Month 11 Complete!
Fetching 12/2025...
✅ Month 12 Complete!

🎉 PIPELINE FINISHED! All 12 months of 2025 data are safely on your hard drive.


In [12]:
import pandas as pd
import sqlite3
import glob

# Create the SQL Database file in your data folder
db_path = "../data/divvy_2025.db"
conn = sqlite3.connect(db_path)

# Find all 12 of the CSV files we just downloaded
csv_files = glob.glob("../data/raw/*-divvy-tripdata.csv")

print("Spinning up SQL Database and loading data...\n")
total_rows = 0

# The "Load" Loop (Memory Safe!)
for file in csv_files:
    month_name = file.split("\\")[-1] if "\\" in file else file.split("/")[-1]
    print(f"Loading {month_name} into SQL...")
    
    # Read the month
    df = pd.read_csv(file)
    
    # Push it into a SQL table called 'trips' (append adds it to the bottom)
    df.to_sql("trips", conn, if_exists="append", index=False)
    
    total_rows += len(df)

print(f"\n✅ DATABASE READY! {total_rows:,} rows successfully loaded into the 'trips' table.")

Spinning up SQL Database and loading data...

Loading 202501-divvy-tripdata.csv into SQL...
Loading 202502-divvy-tripdata.csv into SQL...
Loading 202503-divvy-tripdata.csv into SQL...
Loading 202504-divvy-tripdata.csv into SQL...
Loading 202505-divvy-tripdata.csv into SQL...
Loading 202506-divvy-tripdata.csv into SQL...
Loading 202507-divvy-tripdata.csv into SQL...
Loading 202508-divvy-tripdata.csv into SQL...
Loading 202509-divvy-tripdata.csv into SQL...
Loading 202510-divvy-tripdata.csv into SQL...
Loading 202511-divvy-tripdata.csv into SQL...
Loading 202512-divvy-tripdata.csv into SQL...

✅ DATABASE READY! 5,552,994 rows successfully loaded into the 'trips' table.
